<a href="https://colab.research.google.com/github/minti610/AMAN-KUMAR-PODDAR/blob/main/Data_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from pydantic import BaseModel, ValidationError,Field,EmailStr
from typing import Optional, List
import datetime

# Raw dataset with data quality issues
df = pd.DataFrame({
    "patient_id": [101, 102, 103, 103, 105, 106],
    "name": ["Aman", "  Rahul ", "PRIYA", "PRIYA", None, "Sneha"],
    "gender": ["male", " Male ", "FEMALE", "FEMALE", "female", "N/A"],
    "age": [22, -5, 120, 120, np.nan, 28],
    "email": [
        "aman@gmail.com",
        "rahul123",
        "priya@gmail.com",
        "priya@gmail.com",
        "N/A",
        "sneha@mail.com"
    ],
    "phone": [
        "9876543210",
        "12345",
        "9988776655",
        "9988776655",
        None,
        "9876501234"
    ],
    "admission_date": [
        "2025-01-10",
        "32-13-2024",
        "2024/05/12",
        "2024/05/12",
        "",
        "2025-03-18"
    ]
})


In [ ]:
# convert garbage into N/A and remove duplicate
df = df.drop_duplicates(subset=["patient_id"])
df["patient_id"]=df["patient_id"].fillna("N/A")
df["name"]=df["name"].str.strip().str.title()
df["gender"]=df["gender"].str.strip().str.title()
df = df.drop_duplicates(subset=["email"])
df = df.drop_duplicates(subset=["phone"])
df["admission_date"]=pd.to_datetime(df["admission_date"],errors="coerce").replace("","N/A")
df["age"]=df['age'].fillna(df['age'].median())
df

,patient_id,name,gender,age,email,phone,admission_date
0,101,Aman,Male,22.0,aman@gmail.com,9876543210,2025-01-10
1,102,Rahul,Male,-5.0,rahul123,12345,NaT
2,103,Priya,Female,120.0,priya@gmail.com,9988776655,NaT
4,105,None,Female,25.0,N/A,None,NaT
5,106,Sneha,N/A,28.0,sneha@mail.com,9876501234,2025-03-18


In [ ]:
pip install 'pydantic[email]'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 16.5 MB/s eta 0:00:00


In [ ]:
#validation using pydantic
from datetime import datetime
from pydantic import BaseModel, ValidationError,Field,EmailStr
from typing import Optional, List
import pandas as pd

class paicent(BaseModel):
    patient_id: Optional[int]=Field(gt=0,lt=1000)
    name: Optional[str]=Field(min_length=3)
    gender: Optional[str]=Field(min_length=3)
    age: Optional[int]
    email:Optional[EmailStr]
    phone: Optional[str]=Field(min_length=10,max_length=10)
    admission_date: Optional[datetime]
clean_data = []
invalid_data=[]
for row in df.to_dict(orient='records'):
  try:
    paicent_instance=paicent.model_validate(row)
    clean_data.append(paicent_instance.model_dump())
  except Exception as e:
        invalid_data.append({"data": row, "error": str(e)})
df_clean = pd.DataFrame(clean_data)
df_invalid = pd.DataFrame(invalid_data)
print(df_clean)
print(df_invalid)

   patient_id   name  gender  age            email       phone admission_date
0         101   Aman    Male   22   aman@gmail.com  9876543210     2025-01-10
1         103  Priya  Female  120  priya@gmail.com  9988776655            NaT
2         106  Sneha     N/A   28   sneha@mail.com  9876501234     2025-03-18
                                                data  \
0  {'patient_id': 102, 'name': 'Rahul', 'gender':...   
1  {'patient_id': 105, 'name': None, 'gender': 'F...   

                                               error  
0  2 validation errors for paicent\nemail\n  valu...  
1  1 validation error for paicent\nemail\n  value...  


In [ ]:
df_clean

,patient_id,name,gender,age,email,phone,admission_date
0,101,Aman,Male,22,aman@gmail.com,9876543210,2025-01-10
1,103,Priya,Female,120,priya@gmail.com,9988776655,NaT
2,106,Sneha,N/A,28,sneha@mail.com,9876501234,2025-03-18
